### Step 1: Load 2 text PDFs

In [91]:
# Step 0 (Setup): Install required packages (run once per environment)
%pip install -q langchain langchain-classic langchain-community langchain-openai langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers python-dotenv rank-bm25 gradio pypdf

Note: you may need to restart the kernel to use updated packages.


In [92]:
# Step 1 (Load): Load two text-based PDFs into LangChain Documents

from langchain_community.document_loaders import PyPDFLoader

# Define the PDF file paths (assumed to be in the current working directory)
pdf_paths = ["textbook_1.pdf", "textbook_2.pdf"]

# Load each PDF into a list of Document objects (typically one Document per page)
all_documents = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    docs = loader.load()
    all_documents.extend(docs)

# Print assignment-required outputs
print(f"Number of documents loaded: {len(all_documents)}")

# Print a sample from the first loaded document (first 700 characters)
if all_documents:
    sample_text = all_documents[0].page_content[:700]
    print("\nSample of first document content:\n")
    print(sample_text)
else:
    print("\nNo documents were loaded.")

Number of documents loaded: 1201

Sample of first document content:

Chip Huyen
 AI Engineering
Building Applications  
with Foundation Models


In [93]:
# Step 2 (Chunk): Split loaded documents into chunks with two configurations + metadata
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Prefer the full Step 1 output, but support `docs` if that's your loaded list name
if "all_documents" in globals() and all_documents:
    source_docs = all_documents
elif "docs" in globals() and docs:
    source_docs = docs
else:
    source_docs = []

# Map each textbook to a date tag (adjust these to your real document dates if available)
source_date_map = {
    "textbook_1.pdf": "2023-01-01",
    "textbook_2.pdf": "2024-01-01",
}

# Add metadata fields requested for filtered retrieval
# - source: source file name
# - date: date tag per source document
# - section: derived from page number bucket (chapter/section proxy)
def enrich_chunk_metadata(chunks):
    for chunk in chunks:
        src_path = chunk.metadata.get("source", "")
        source_file = os.path.basename(src_path) if src_path else "unknown_source"
        page_num = int(chunk.metadata.get("page", -1)) + 1

        if page_num <= 0:
            section = "unknown_section"
        else:
            section_start = ((page_num - 1) // 20) * 20 + 1
            section_end = section_start + 19
            section = f"pages_{section_start}_{section_end}"

        chunk.metadata["source"] = source_file
        chunk.metadata["date"] = source_date_map.get(source_file, "unknown_date")
        chunk.metadata["section"] = section
    return chunks

# Helper to split documents and print assignment-required chunk statistics
def chunk_and_report(documents, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(documents)
    chunks = enrich_chunk_metadata(chunks)

    chunk_lengths = [len(chunk.page_content) for chunk in chunks]
    min_len = min(chunk_lengths) if chunk_lengths else 0
    max_len = max(chunk_lengths) if chunk_lengths else 0

    print(f"Configuration: chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")
    print(f"Total number of chunks created: {len(chunks)}")
    print(f"Character count of the smallest chunk: {min_len}")
    print(f"Character count of the largest chunk: {max_len}")

    if chunks:
        sample_meta = {k: chunks[0].metadata.get(k) for k in ["source", "date", "section"]}
        print(f"Sample metadata from first chunk: {sample_meta}")

    print("-" * 60)
    return chunks

# First required configuration
chunks_500_100 = chunk_and_report(source_docs, chunk_size=500, chunk_overlap=100)

# Second required configuration
chunks_1000_150 = chunk_and_report(source_docs, chunk_size=1000, chunk_overlap=150)

Configuration: chunk_size=500, chunk_overlap=100
Total number of chunks created: 4979
Character count of the smallest chunk: 2
Character count of the largest chunk: 500
Sample metadata from first chunk: {'source': 'textbook_1.pdf', 'date': '2023-01-01', 'section': 'pages_1_20'}
------------------------------------------------------------
Configuration: chunk_size=1000, chunk_overlap=150
Total number of chunks created: 2571
Character count of the smallest chunk: 2
Character count of the largest chunk: 1000
Sample metadata from first chunk: {'source': 'textbook_1.pdf', 'date': '2023-01-01', 'section': 'pages_1_20'}
------------------------------------------------------------


In [94]:
# Step 3 (Embed + Store): Embed 500-size chunks and persist them in ChromaDB
import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Use the smaller chunk set from Step 2
if "chunks_500_100" not in globals() or not chunks_500_100:
    raise ValueError("`chunks_500_100` is missing. Run Step 2 first.")

# Initialize embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create persistent ChromaDB directory
persist_dir = "./chroma_db"
os.makedirs(persist_dir, exist_ok=True)

# Recreate the collection each run to avoid duplicate vectors from repeated executions
try:
    existing_store = Chroma(
        collection_name="textbook_rag",
        embedding_function=embedding_model,
        persist_directory=persist_dir,
    )
    existing_store.delete_collection()
except Exception:
    pass

vectorstore = Chroma.from_documents(
    documents=chunks_500_100,
    embedding=embedding_model,
    collection_name="textbook_rag",
    persist_directory=persist_dir,
)

# Print assignment-required stats
num_vectors = vectorstore._collection.count()
sample_embedding = embedding_model.embed_query("sample text")

print(f"Number of vectors stored: {num_vectors}")
print(f"Sample embedding shape: {len(sample_embedding)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6986.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of vectors stored: 4979
Sample embedding shape: 384


In [95]:
# Step 4 (Test Retrieval): Query existing ChromaDB before building a RAG chain
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Reconnect to the same embedding model and persisted Chroma collection
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    collection_name="textbook_rag",
    embedding_function=embedding_model,
    persist_directory="./chroma_db",
)

# Placeholder test queries (replace with your assignment-specific questions later)
test_queries = [
    "What is the definition of X?",
    "Explain concept Y.",
    "How does Z work?",
]

# Helper for a simple relevance note based on keyword overlap
stop_words = {"what", "is", "the", "of", "explain", "how", "does", "work", "concept", "definition"}

def relevance_note(query, chunk_text):
    query_terms = {w.lower().strip(".,?!:;()[]{}\"'") for w in query.split()}
    query_terms = {w for w in query_terms if w and w not in stop_words and len(w) > 1}
    chunk_lower = chunk_text.lower()

    matches = [term for term in query_terms if term in chunk_lower]
    if matches:
        return f"Relevant: Yes — because it contains query term(s): {', '.join(matches)}"
    return "Relevant: No — because there is no clear keyword overlap with the query."

# Run top-3 similarity retrieval for each query and print readable output
for query in test_queries:
    print("=" * 90)
    print(f"Query: {query}")

    results = vectorstore.similarity_search(query, k=3)
    for i, doc in enumerate(results, start=1):
        chunk_text = doc.page_content.strip()
        preview = (chunk_text[:300] + "...") if len(chunk_text) > 300 else chunk_text

        print(f"\nResult {i} (trimmed):")
        print(preview)
        print(relevance_note(query, chunk_text))

print("\nRetrieval test complete. (No LLM calls, no RAG chain built.)")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7916.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: What is the definition of X?

Result 1 (trimmed):
3 While I love writing, one of the things I absolutely do not enjoy is trying to condense everyone’s opinions into
one single definition. IBM defined data quality along seven dimensions: completeness, uniqueness, validity,
timeliness, accuracy, consistency, and fitness for purpose. Wikipedia added a...
Relevant: No — because there is no clear keyword overlap with the query.

Result 2 (trimmed):
Your goal is to predict what someone with 10 years of
experience should earn. In this example, x = 10, and you want
to predict what y should be.
Relevant: No — because there is no clear keyword overlap with the query.

Result 3 (trimmed):
a 5% chance that the point corresponds to class 1. But if x =
10, there is about a 76% chance that it’s class 1. If asked
to classify that point as a red or a blue, you would conclude
that it’s a red because it’s much more likely to be red
than blue.
Relevant: No — because there is no clear keyword overla

In [96]:
# Step 5 (Build RAG Chain): Hybrid retrieval + metadata-filtered retrieval + RetrievalQA
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_openai import ChatOpenAI

# EnsembleRetriever location differs across LangChain packaging layouts
try:
    from langchain.retrievers import EnsembleRetriever
except ModuleNotFoundError:
    from langchain_classic.retrievers import EnsembleRetriever

# Compatibility imports for different LangChain package layouts
try:
    from langchain.chains import RetrievalQA
    from langchain.prompts import ChatPromptTemplate
except ModuleNotFoundError:
    from langchain_classic.chains import RetrievalQA
    from langchain_core.prompts import ChatPromptTemplate

# Load environment variables from .env (works in notebooks even when cwd differs)
load_dotenv()
project_env = Path.cwd() / ".env"
if project_env.exists():
    load_dotenv(project_env, override=False)

# Reconnect to the same embedding model and persisted Chroma collection from Step 3
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = Chroma(
    collection_name="textbook_rag",
    embedding_function=embedding_model,
    persist_directory="./chroma_db",
)

# Use Step 2 chunks for BM25 and metadata filtering
if "chunks_500_100" not in globals() or not chunks_500_100:
    raise ValueError("Run Step 2 first so `chunks_500_100` exists.")

# Optional metadata filters for retrieval. Set to None to disable a filter.
active_filters = {
    "source": None,              # Example: "textbook_1.pdf"
    "section": None,             # Example: "pages_1_20"
    "date": None,                # Example: "2023-01-01"
}

# Build a Chroma filter dict from enabled filters
def build_chroma_filter(filters):
    return {k: v for k, v in filters.items() if v is not None}

# Apply filters to in-memory chunks (used by BM25)
def filter_documents_by_metadata(documents, filters):
    filtered = []
    for d in documents:
        keep = True
        for k, v in filters.items():
            if v is not None and d.metadata.get(k) != v:
                keep = False
                break
        if keep:
            filtered.append(d)
    return filtered

active_filter_dict = build_chroma_filter(active_filters)
filtered_docs_for_bm25 = filter_documents_by_metadata(chunks_500_100, active_filters)

if not filtered_docs_for_bm25:
    raise ValueError("Metadata filters returned 0 docs. Relax `active_filters` and rerun Step 5.")

# Vector retriever with metadata filtering
vector_search_kwargs = {"k": 3}
if active_filter_dict:
    vector_search_kwargs["filter"] = active_filter_dict

retriever_vector = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs=vector_search_kwargs,
)

# BM25 retriever on the same filtered subset
retriever_bm25 = BM25Retriever.from_documents(filtered_docs_for_bm25)
retriever_bm25.k = 3

# Combine retrievers: tune weights for more keyword vs semantic influence
retriever = EnsembleRetriever(
    retrievers=[retriever_vector, retriever_bm25],
    weights=[0.5, 0.5],
)

# Quick retrieval comparison on Step 4 queries
queries_compare = test_queries if "test_queries" in globals() else [
    "What is the definition of X?",
    "Explain concept Y.",
    "How does Z work?",
]

def _doc_fingerprint(doc):
    return hash((doc.page_content or "").strip())

print(f"Active metadata filter: {active_filter_dict if active_filter_dict else 'None'}")
print(f"BM25 candidate docs after filtering: {len(filtered_docs_for_bm25)}")
print("Hybrid retrieval check (vector vs ensemble top-3 overlap):")
for q in queries_compare:
    v_docs = retriever_vector.invoke(q)
    h_docs = retriever.invoke(q)
    v_fps = {_doc_fingerprint(d) for d in v_docs}
    h_fps = {_doc_fingerprint(d) for d in h_docs}
    overlap = len(v_fps & h_fps)
    print(f"- Query: {q!r} | overlap={overlap}/3")
print("-" * 90)

# Define a chat prompt with an explicit system message
custom_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a warm, friendly, and helpful assistant. "
        "If the user message is only a greeting (like hi, hey, or hello), respond with a short friendly greeting and ask how you can help. "
        "For non-greeting questions, use ONLY the provided context. "
        "If the answer is not in the context, say: 'I don't know based on the provided context.'",
    ),
    (
        "human",
        "CONTEXT:\nThe information in the two PDF documents textbook_1.pdf and textbook_2.pdf.\n{context}\n\n"
        "QUESTION:\n{question}\n\nANSWER:",
    ),
])

# Initialize LLM after loading .env
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is not set. Confirm .env is in the notebook working directory and restart/re-run this cell.")

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

# Build RetrievalQA chain using 'stuff' strategy and custom prompt
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=False,
    chain_type_kwargs={"prompt": custom_prompt},
)

# Reuse Step 4 queries when available; otherwise default placeholders
queries = queries_compare

# Run the same 3 queries and print answers cleanly
for query in queries:
    result = qa_chain.invoke({"query": query})
    answer = result["result"] if isinstance(result, dict) else str(result)

    print(f"Query: {query}")
    print(f"Answer: {answer}")
    print("-" * 90)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2939.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Active metadata filter: None
BM25 candidate docs after filtering: 4979
Hybrid retrieval check (vector vs ensemble top-3 overlap):
- Query: 'What is the definition of X?' | overlap=3/3
- Query: 'Explain concept Y.' | overlap=3/3
- Query: 'How does Z work?' | overlap=3/3
------------------------------------------------------------------------------------------
Query: What is the definition of X?
Answer: I don't know based on the provided context.
------------------------------------------------------------------------------------------
Query: Explain concept Y.
Answer: I don't know based on the provided context.
------------------------------------------------------------------------------------------
Query: How does Z work?
Answer: I don't know based on the provided context.
------------------------------------------------------------------------------------------


In [97]:
# Step 6 (Evaluation): Evaluate retrieval relevance, grounding, and answer correctness

# Ensure Step 5 objects exist
if "retriever" not in globals() or "qa_chain" not in globals():
    raise ValueError("Run Step 5 first so `retriever` and `qa_chain` are available.")

# Small evaluation set (replace placeholders with your real questions/answers)
eval_set = [
    {
        "question": "What is concept A?",
        "gold_answer": "[Replace with known correct answer for concept A]",
        "reference_keywords": ["concept", "a"],
    },
    {
        "question": "How does method B work?",
        "gold_answer": "[Replace with known correct answer for method B]",
        "reference_keywords": ["method", "b"],
    },
    {
        "question": "Define term C.",
        "gold_answer": "[Replace with known correct definition for term C]",
        "reference_keywords": ["term", "c", "define"],
    },
    {
        "question": "What is the purpose of D?",
        "gold_answer": "[Replace with known purpose of D]",
        "reference_keywords": ["purpose", "d"],
    },
    {
        "question": "Explain relationship between E and F.",
        "gold_answer": "[Replace with known relationship between E and F]",
        "reference_keywords": ["relationship", "e", "f"],
    },
]

# Lightweight helper to normalize text for simple heuristic checks
def normalize(text):
    return " ".join(text.lower().split())

results = []
retrieval_hits = 0
faithfulness_hits = 0
correctness_hits = 0

# Evaluate each question: retriever-only first, then full RAG chain
for item in eval_set:
    question = item["question"]
    gold_answer = item["gold_answer"]
    keywords = [k.lower() for k in item["reference_keywords"]]

    # 1) Retriever-only pass
    retrieved_docs = retriever.invoke(question)
    retrieved_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    retrieved_text_norm = normalize(retrieved_text)

    retrieval_relevant = any(k in retrieved_text_norm for k in keywords)
    retrieval_hits += int(retrieval_relevant)

    # 2) Full RAG chain pass
    rag_output = qa_chain.invoke({"query": question})
    answer = rag_output["result"] if isinstance(rag_output, dict) else str(rag_output)
    answer_norm = normalize(answer)

    # 3a) Faithfulness (grounded): answer terms should appear in retrieved context
    answer_tokens = [t for t in answer_norm.replace(".", " ").replace(",", " ").split() if len(t) > 3]
    overlap_count = sum(1 for t in set(answer_tokens) if t in retrieved_text_norm)
    grounded_in_context = overlap_count >= max(1, min(3, len(set(answer_tokens)) // 5 + 1))
    faithfulness_hits += int(grounded_in_context)

    # 3b) Correctness: heuristic check against user-known answer placeholder
    # Replace this logic later with your own strict grading based on real expected answers.
    if gold_answer.startswith("[Replace with"):
        answer_correct = False
    else:
        gold_norm = normalize(gold_answer)
        answer_correct = gold_norm in answer_norm or answer_norm in gold_norm
    correctness_hits += int(answer_correct)

    results.append(
        {
            "question": question,
            "retrieval_relevant": retrieval_relevant,
            "grounded_in_context": grounded_in_context,
            "answer_correct": answer_correct,
            "answer": answer,
        }
    )

# Print structured evaluation output
print("=" * 110)
print("Step 6 Evaluation Results")
print("=" * 110)
for idx, row in enumerate(results, start=1):
    print(f"\nQ{idx}: {row['question']}")
    print(f"Retrieved relevant chunks: {'Yes' if row['retrieval_relevant'] else 'No'}")
    print(f"Answer grounded in context: {'Yes' if row['grounded_in_context'] else 'No'}")
    print(f"Answer correct: {'Yes' if row['answer_correct'] else 'No'}")
    print(f"Generated answer: {row['answer']}")
    print("-" * 110)

# Compute and print assignment-required metrics
n = len(eval_set)
print(f"Retrieval accuracy: {retrieval_hits}/{n}")
print(f"Faithfulness accuracy: {faithfulness_hits}/{n}")
print(f"Correctness accuracy: {correctness_hits}/{n}")

Step 6 Evaluation Results

Q1: What is concept A?
Retrieved relevant chunks: Yes
Answer grounded in context: No
Answer correct: No
Generated answer: I don't know based on the provided context.
--------------------------------------------------------------------------------------------------------------

Q2: How does method B work?
Retrieved relevant chunks: Yes
Answer grounded in context: Yes
Answer correct: No
Generated answer: I don't know based on the provided context.
--------------------------------------------------------------------------------------------------------------

Q3: Define term C.
Retrieved relevant chunks: Yes
Answer grounded in context: Yes
Answer correct: No
Generated answer: The term C in the given context refers to a parameter that controls how aggressively a model fits to the training data in machine learning algorithms.
--------------------------------------------------------------------------------------------------------------

Q4: What is the purpose of D?

In [98]:
# Optional UI: Test the RAG chain with vector vs hybrid retrieval debug in Gradio
import gradio as gr

# Ensure Step 5 has been executed so `qa_chain`, `retriever`, and `retriever_vector` exist
if "qa_chain" not in globals() or "retriever" not in globals() or "retriever_vector" not in globals():
    raise ValueError("Run Step 5 first so `qa_chain`, `retriever`, and `retriever_vector` are available.")

# Function used by the UI to query the RAG chain and inspect retrieved context
def ask_rag(question):
    question = (question or "").strip()
    if not question:
        return "Please enter a question.", "", ""

    # Handle greetings directly for a friendlier chat experience
    normalized = question.lower().strip(".!? ")
    greetings = {"hi", "hey", "hello", "hiya", "yo"}
    if normalized in greetings:
        return "Hi! What can I help you with today?", "", ""

    vector_docs = retriever_vector.invoke(question)
    hybrid_docs = retriever.invoke(question)

    result = qa_chain.invoke({"query": question})
    answer = result["result"] if isinstance(result, dict) else str(result)

    def format_docs(docs, label):
        parts = [f"{label} (top {len(docs)})"]
        for i, doc in enumerate(docs, start=1):
            text = doc.page_content.strip()
            preview = (text[:300] + "...") if len(text) > 300 else text
            meta = {
                "source": doc.metadata.get("source"),
                "date": doc.metadata.get("date"),
                "section": doc.metadata.get("section"),
            }
            parts.append(f"Chunk {i} | metadata={meta}:\n{preview}")
        return "\n\n".join(parts)

    return (
        answer,
        format_docs(vector_docs, "Vector only"),
        format_docs(hybrid_docs, "Hybrid (vector + BM25)"),
    )

# Build and launch a minimal interface (question -> answer + retrieved chunks)
demo = gr.Interface(
    fn=ask_rag,
    inputs=gr.Textbox(lines=3, label="Ask a question"),
    outputs=[
        gr.Textbox(lines=8, label="RAG Answer"),
        gr.Textbox(lines=14, label="Retrieved Chunks: Vector only"),
        gr.Textbox(lines=14, label="Retrieved Chunks: Hybrid"),
    ],
    title="Textbook RAG Assistant",
    description="Ask questions about textbook_1.pdf and textbook_2.pdf using your RetrievalQA chain.",
)

demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
